# 머신러닝 파일별 보강2 · 4-3 건포도 품종 분류 과제 풀이

**한 줄 요약**

- 건포도 900개의 크기
- 모양 측정값으로 Kecimen / Besni를 가르는 과제 · 측정값 2개 고르기 · 로지스틱 vs KNN · 스케일링 · k 고르기 · 최종 1개 · test 딱 한 번

- **원본** · `6.머닝러신/05_건포도_분류과제/4-3_건포도과제.ipynb` (원본 파일은 그대로 둠)
- **종합 정리본** · `00_최소선/05_머신러닝` 머신러닝-2 · 3 (스케일링 · 3분할) · 파일별 5 · 보강 1과 같은 절차

## 🎯 학습 목표

이 노트북을 끝내면 **자료를 보지 않고** 다음을 할 수 있어야 함

1. 측정값 7개 중 2개를 train 안의 근거(분리 정도 · CV)로 고르고 이유를 말한다
2. 같은 2개로 로지스틱과 KNN(k=5)을 validation에서 비교하고 기준선 0.5와 비교한다
3. 스케일링 전후 KNN의 정확도와 '예측이 바뀐 개수'를 근거로 쓴다
4. train 내부 CV로 k를 고르고, 네 후보 중 하나를 근거 2개로 고른다
5. 고른 모델을 dev 720개로 다시 학습해 test 180개를 딱 한 번 평가하고 한계를 말한다

## 어디에 쓰나

*실제 상황 → 이 권에서 배우는 것*

- **측정값이 많은데 몇 개만 쓸 수 있을 때** · train 안의 분리 정도 · CV로 고르기 (1장)
- **단위가 다른 측정값이 섞여 있을 때** · KNN 스케일링 · 바뀐 예측 수 (2-2)
- **후보 모델 여러 개 중 하나를 골라야 할 때** · validation 표 + 정확도 외의 근거 (3-2)
- **보고서에 최종 성능을 적을 때** · dev 재학습 → test 1회 (4장)

## 학습 지도

<img src="images/pfm9_map.png" alt="4-3 건포도 학습 지도" width="763">

**예상 학습 시간** · 개념 30분 + 실습 1시간 (전체 실행 30초 이내 · GPU 필요 없음)

## 목차

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">장</th><th style="text-align: left;">제목</th><th style="text-align: left;">이 장의 질문</th></tr>
</thead>
<tbody>
<tr><td>0</td><td>준비</td><td>건포도 900개 · 원본 과제</td></tr>
<tr><td>1</td><td>측정값 2개 고르기</td><td>측정값 7개 중 어떤 2개가 두 품종을 덜 겹치게 가르는가?</td></tr>
<tr><td>2</td><td>로지스틱 vs KNN · 스케일링</td><td>같은 2개로 만든 로지스틱과 KNN은 validation에서 어떻게 다르고, 스케일링은 KNN의 무엇을 바꾸는가?</td></tr>
<tr><td>3</td><td>k 고르기 · 최종 1개</td><td>k는 무엇으로 고르고, 네 후보 중 하나는 어떤 근거 2개로 고르는가?</td></tr>
<tr><td>4</td><td>종합 실습</td><td>고른 모델을 dev 720개로 다시 학습해 test를 한 번 평가하면 validation과 얼마나 다른가?</td></tr>
<tr><td>—</td><td>마무리 · 부록</td><td>오늘 배운 것 · 기억 확인 · 오답 노트 · 면접 · 요약표 · 복습일 · 원본 대응표 · 전체 코드</td></tr>
</tbody></table>

---
## 0. 준비

원본 폴더의 `raisin.csv`를 읽음 (원본은 UCI에서 받고 인터넷이 안 되면 이 파일을 씀 · 같은 900행). 원본은 빈칸 과제라 이 노트북은 **직접 푼 한 가지 답**. **셀은 위에서부터 차례로 실행**.

*항목 → 설정*

- **라이브러리** · numpy · pandas · scikit-learn
- **seed** · 분할 `random_state=42` 두 번 (원본과 같음)
- **데이터** · `../05_건포도_분류과제/raisin.csv` · 900행 × 측정값 7개 · Kecimen 450 · Besni 450
- **출처** · UCI Machine Learning Repository — Raisin (id 850) · CC BY 4.0

**실행 흐름**

1. 모델 · 평가 도구 불러오기
2. 원본 폴더의 건포도 CSV 읽기
3. test 180개 봉인 · train 576 / validation 144 분할

In [1]:
# 준비: 건포도 900개 · train 576 / validation 144 / test 180 (원본 셀 2와 같은 분할)
# 왜 · 원본 과제와 같은 3분할로 결과를 그대로 비교하기 위해
import itertools  # 측정값 2개 조합

import numpy as np                                         # 배열
import pandas as pd                                        # 표
from sklearn.linear_model import LogisticRegression        # 로지스틱
from sklearn.metrics import accuracy_score                 # 정확도
# 탐색 · CV · 분할
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier as KNN  # KNN
from sklearn.pipeline import Pipeline                      # 전처리 + 모델
from sklearn.preprocessing import StandardScaler           # 스케일링

# 왜 · 원본 폴더의 데이터를 상대 경로로 읽고, test는 처음부터 떼어 봉인하기 위해
df = pd.read_csv("../05_건포도_분류과제/raisin.csv")  # 원본은 UCI에서 받고 실패하면 이 파일
# test 봉인
dev, test_df = train_test_split(df, test_size=0.2, stratify=df["Class"], random_state=42)
train_df, valid_df = train_test_split(                # dev를 train · validation으로
    dev, test_size=0.2, stratify=dev["Class"], random_state=42)
y_train, y_valid, y_test = train_df["Class"], valid_df["Class"], test_df["Class"]  # 정답
FEATURES = [c for c in df.columns if c != "Class"]    # 측정값 7개
# 576 / 144 / 180
print("train", len(train_df), "/ validation", len(valid_df), "/ test", len(test_df))
print("class", df["Class"].value_counts().to_dict(), "· baseline(찍기) 0.5")  # 반반

train 576 / validation 144 / test 180
class {'Kecimen': 450, 'Besni': 450} · baseline(찍기) 0.5


### 원본 과제 · 지켜야 할 것

- 건포도 900개의 크기 · 모양 측정값으로 두 품종(**Kecimen** / **Besni**)을 구분하는 모델 만들기

*# → 문제*

- **1** · 측정값 7개 중 **2개를 골라** 근거 적기
- **2** · 그 2개로 **로지스틱 회귀**와 **KNN(k=5)** 을 만들고 validation 정확도 비교
- **3** · KNN을 **스케일링해서 다시** 재고 무엇이 달라졌는지 적기
- **4** · KNN의 **k를 train 내부 CV로** 고르기
- **5** · 최종 모델 **하나**를 고르고 근거 적기
- **6** · **그다음에** test를 **딱 한 번** 평가

**원본 힌트**

- 문제 6 전에는 test를 열지 않기 · test 점수를 보고 모델을 바꾸지 않기 (3-2 · 4-1 · 4-2에서 계속 한 얘기)
- 정답은 하나로 정해져 있지 않음 · 이 노트북은 직접 푼 한 가지 답 · 원본 폴더의 정답 · 상세 설명편은 직접 푼 뒤에 비교

---
## 1. 측정값 2개 고르기

> **이 장의 흐름**
>
> - train 576개로만 측정값마다 두 품종이 얼마나 떨어져 있는지 확인
> - 그다음 2개 조합 21개를 train 내부 CV로 줄 세워 하나를 고름

<img src="images/pfm9_pair.png" alt="train 산점도 · 고른 두 측정값" width="543">

**파이프라인 위치** · ③ 특성 선택 (train만)

### 1-1. `A` 측정값 하나씩 · 분리 정도 · `groupby().mean()` · `분리 점수` <sub>원본 셀 3 ~ 5</sub>

> 🎯 **측정값 7개 각각에서 두 품종의 평균은 얼마나 떨어져 있는가?**

💡 **쉽게 말하면**

- 두 품종의 점이 **덜 겹치는** 측정값이 좋은 단서.
- 평균 차이 ÷ 표준편차가 크면 덜 겹침 (train만 봄).

**실행 흐름**

1. train에서 품종별 평균
2. 평균 차이 ÷ 두 표준편차의 평균
3. 큰 순서로 정렬

**먼저 예측** · 가장 잘 가르는 측정값은 크기(면적 · 길이) 쪽일까 모양(Extent · Eccentricity) 쪽일까? — 답은 코드 실행 결과 아래에

In [2]:
# 왜 · 두 품종을 가장 잘 가르는 특성을 평균 차이 · 퍼짐으로 미리 살펴보기 위해
g = train_df.groupby("Class")[FEATURES]                        # train만 · 품종별
gap = (g.mean().loc["Besni"] - g.mean().loc["Kecimen"]).abs()  # 평균 차이
spread = g.std().mean()                                        # 두 품종 표준편차의 평균
sep = (gap / spread).sort_values(ascending=False)              # 클수록 덜 겹침
print(sep.round(3).to_string())                                # 분리 점수
# 거의 같은 정보
print("크기 측정값끼리 상관:", round(train_df[["Area", "ConvexArea"]].corr().iloc[0, 1], 3))

MajorAxisLength    1.916
Perimeter          1.847
Area               1.753
ConvexArea         1.745
MinorAxisLength    1.201
Eccentricity       1.009
Extent             0.264
크기 측정값끼리 상관: 0.995


**결과** · 예측 정답은 크기 쪽 — MajorAxisLength 1.916 · Extent는 0.264로 꼴찌

- `MajorAxisLength    1.916` — 분리 점수 1위 · Besni가 더 김
- `Perimeter          1.847` — 크기 측정값들이 상위권
- `Extent             0.264` — 하나만 보면 가장 약한 단서
- `크기 측정값끼리 상관: 0.995` — Area와 ConvexArea는 거의 같은 정보 → 둘을 함께 고르면 손해

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>분리 점수 (평균 차이 ÷ 표준편차)</td><td>측정값 하나씩</td><td>후보를 줄일 때</td></tr>
<tr><td>산점도 (원본 plot_pair)</td><td>두 개 조합의 겹침을 눈으로</td><td>조합을 고를 때</td></tr>
<tr><td>train 내부 CV (1-2)</td><td>조합의 성능을 숫자로</td><td>최종 근거</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - validation · test는 아직 보면 안 됨 → 분리 정도도 train으로만
> - 상관이 큰 두 측정값(Area · ConvexArea)은 정보가 겹침

> ✅ **한 줄 정리** · 측정값마다 두 품종의 평균 차이를 표준편차로 나눠 보면 어떤 단서가 덜 겹치는지 train만으로 알 수 있다.

**딥러닝 코드에서는**

- 입력 채널마다 클래스별 분포를 먼저 보는 EDA 습관

### 1-2. `A` 2개 조합 21개를 train CV로 · `itertools.combinations` · `cross_val_score` <sub>원본 셀 4 · 5 (문제 1)</sub>

> 🎯 **2개 조합 21개 중 train 내부 CV가 가장 높은 조합은? 힌트의 Perimeter + Eccentricity는 몇 위인가?**

💡 **쉽게 말하면**

- 하나씩 좋은 측정값 두 개가 **함께 좋은** 조합은 아닐 수 있음 (정보가 겹치면).
- 그래서 조합 자체를 train 안에서 CV로 잼.

**실행 흐름**

1. 21개 조합
2. 스케일러 + 로지스틱 · train 5-Fold
3. 순위 · 1위 조합 고정

**먼저 예측** · 1위 조합은 분리 점수 1 · 2위를 합친 것일까? — 답은 코드 실행 결과 아래에

In [3]:
# 왜 · 특성 두 개 조합마다 같은 모델로 점수를 비교해 힌트의 근거를 찾기 위해
lr5000 = LogisticRegression(max_iter=5000)                                # 원본 설정
pair_lr = Pipeline([("scaler", StandardScaler()), ("lr", lr5000)])        # 조합 비교용
rows = []                                                                 # 조합마다
for a, b in itertools.combinations(FEATURES, 2):                          # 7개 중 2개 = 21개
    s = cross_val_score(pair_lr, train_df[[a, b]], y_train, cv=5).mean()  # train 안에서만
    rows.append([f"{a} + {b}", s])                                        # 한 줄
pairs = pd.DataFrame(rows, columns=["조합", "train CV"])                  # 21행
pairs = pairs.sort_values("train CV", ascending=False)                    # 1위부터
print(pairs.head(5).round(4).to_string(index=False))                      # 위 5개
# 힌트 조합 순위
hint_rank = int((pairs["조합"] == "Eccentricity + Perimeter").to_numpy().argmax()) + 1
print("힌트 조합(Perimeter + Eccentricity) 순위:", hint_rank, "/ 21")     # 참고
PAIR = pairs.iloc[0]["조합"].split(" + ")                                 # 1위 조합 고정
X_train, X_valid = train_df[PAIR], valid_df[PAIR]                         # 뒤 문제에서 계속 씀
print("고른 측정값:", PAIR)                                               # 문제 1 답

                          조합  train CV
    MajorAxisLength + Extent    0.8663
   Eccentricity + ConvexArea    0.8611
          Extent + Perimeter    0.8593
MajorAxisLength + ConvexArea    0.8576
 MajorAxisLength + Perimeter    0.8576
힌트 조합(Perimeter + Eccentricity) 순위: 6 / 21
고른 측정값: ['MajorAxisLength', 'Extent']


**결과** · 예측 정답은 아님 — MajorAxisLength(1위) + Extent(꼴찌)

- `MajorAxisLength + Extent    0.8663` — 1위 · 크기 하나 + 모양 하나
- `Eccentricity + ConvexArea    0.8611` — 2위 · 차이 0.005
- `힌트 조합(Perimeter + Eccentricity) 순위: 6 / 21` — 원본 힌트 예시는 6위
- `고른 측정값: ['MajorAxisLength', 'Extent']` — 서술 1 · 하나씩은 꼴찌인 Extent가 크기와 겹치지 않는 정보를 더함

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>하나씩 좋은 두 개</td><td>정보가 겹칠 수 있음 (Area + ConvexArea)</td><td>조합 순위가 낮을 수 있음</td></tr>
<tr><td>크기 + 모양</td><td>서로 다른 정보</td><td>1위 조합</td></tr>
<tr><td>train CV 순위</td><td>조합끼리 차이가 작음 (0.866 vs 0.861)</td><td>1위를 '정답'이라 부르지 않기</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 1 · 2위 차이가 작아 다른 조합을 골라도 틀린 답이 아님 (원본: 정답이 하나로 정해져 있지 않음)
> - 조합 선택에 validation을 쓰지 않음 → validation은 문제 2 ~ 5의 모델 비교용

> ✅ **한 줄 정리** · 측정값 2개는 train 안의 CV로 조합 자체를 비교해 고르며, 이번에는 크기(MajorAxisLength)와 모양(Extent)을 하나씩 묶은 조합이 1위였다.

**딥러닝 코드에서는**

- 입력 조합 · 채널 선택도 train · val 안에서만 비교

---
## 2. 로지스틱 vs KNN · 스케일링

> **이 장의 흐름**
>
> - 로지스틱 · KNN(k=5)을 validation에서 비교하고 기준선 0.5와 비교
> - 그다음 KNN을 스케일링해 다시 재고 예측이 바뀐 건포도 수를 셈

<img src="images/pfm9_scale.png" alt="MajorAxisLength(수백) vs Extent(0 ~ 1) · 스케일링 전후 KNN" width="550">

**파이프라인 위치** · ④ 모델 → ③ 전처리

### 2-1. `A` 로지스틱 vs KNN(k=5) · `LogisticRegression` · `KNeighborsClassifier` <sub>원본 셀 6 ~ 8 (문제 2)</sub>

> 🎯 **고른 2개로 로지스틱과 KNN(k=5)을 만들면 validation 정확도는? 기준선 0.5보다 얼마나 높은가?**

💡 **쉽게 말하면**

- 두 모델 모두 같은 2개 · 같은 train으로.
- 기준선 = 반반이라 찍으면 0.5.

**실행 흐름**

1. 로지스틱(max_iter=5000) fit
2. KNN(k=5) fit (스케일링 안 함)
3. validation 정확도 · 예측 저장

**먼저 예측** · 스케일링 없는 KNN은 로지스틱보다 높을까? — 답은 코드 실행 결과 아래에

In [4]:
# 왜 · 스케일링 없이 기준 모델의 validation 점수를 먼저 남기기 위해
lr = LogisticRegression(max_iter=5000).fit(X_train, y_train)                    # 원본 힌트 그대로
lr_pred = lr.predict(X_valid)                                                   # validation 예측
knn = KNN(n_neighbors=5).fit(X_train, y_train)                                  # 스케일링 안 함
knn_pred = knn.predict(X_valid)                                                 # 문제 3에서 비교
acc_lr, acc_knn = accuracy_score(y_valid, lr_pred), accuracy_score(y_valid, knn_pred)  # 두 점수
print(f"로지스틱 validation: {acc_lr:.4f} · 기준선보다 +{acc_lr - 0.5:.4f}")    # 문제 2
print(f"KNN(k=5) validation: {acc_knn:.4f} · 기준선보다 +{acc_knn - 0.5:.4f}")  # 문제 2

로지스틱 validation: 0.8472 · 기준선보다 +0.3472
KNN(k=5) validation: 0.8056 · 기준선보다 +0.3056


**결과** · 예측 정답은 아님 — 로지스틱 0.8472 > KNN 0.8056

- `로지스틱 validation: 0.8472 · 기준선보다 +0.3472` — 기준선보다 0.35 가까이 높음
- `KNN(k=5) validation: 0.8056 · 기준선보다 +0.3056` — 스케일링 없는 KNN이 6개 더 틀림

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>로지스틱</td><td>가중합 → 확률 · 단위 차이를 계수가 흡수</td><td>기준선 모델</td></tr>
<tr><td>KNN (스케일링 없음)</td><td>거리 → 큰 단위 측정값이 지배</td><td>문제 3에서 다시</td></tr>
<tr><td>기준선 0.5</td><td>반반 찍기</td><td>두 모델이 넘어야 할 선</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - validation 144개 → 1개 = 0.0069 · 작은 차이로 우열을 정하지 않기

> ✅ **한 줄 정리** · 같은 2개 측정값으로 로지스틱과 KNN을 같은 validation에서 비교하고, 둘 다 기준선 0.5와도 비교한다.

**딥러닝 코드에서는**

- 새 문제는 단순한 기준선 모델부터 돌려 넘어야 할 선을 정함

### 2-2. `A` 스케일링한 KNN · 바뀐 예측 수 · `fit_transform` · `transform` · `knn_pred != pred_s` <sub>원본 셀 9 ~ 11 (문제 3)</sub>

> 🎯 **스케일링한 뒤 KNN(k=5)의 정확도와 예측이 바뀐 건포도 수는?**

💡 **쉽게 말하면**

- MajorAxisLength는 **수백**, Extent는 **0 ~ 1** → 스케일링 전 거리는 사실상 길이만 봄.
- train에는 fit_transform, validation에는 transform (3-1 규율).

**실행 흐름**

1. scaler를 train에 fit
2. train · validation 변환
3. KNN(k=5) · 정확도 · 바뀐 예측 수

**먼저 예측** · 스케일링하면 예측이 몇 개나 바뀔까? — 답은 코드 실행 결과 아래에

In [5]:
# 왜 · train에만 fit해 validation · test 정보가 새지 않게 스케일링하기 위해
sc = StandardScaler()                                              # 평균 0 · 표준편차 1
X_train_s = sc.fit_transform(X_train)                              # train에만 fit
X_valid_s = sc.transform(X_valid)                                  # validation은 transform만
knn_s = KNN(n_neighbors=5).fit(X_train_s, y_train)                 # 넣는 데이터만 바뀜
pred_s = knn_s.predict(X_valid_s)                                  # 스케일링 후 예측
acc_knn_s = accuracy_score(y_valid, pred_s)                        # 정확도
print(f"KNN(k=5) 스케일링 전 {acc_knn:.4f} → 후 {acc_knn_s:.4f}")  # 서술 2
print(f"예측이 바뀐 건포도: {(knn_pred != pred_s).sum()}개 / {len(pred_s)}")  # 정확도 말고 근거
# 단위 차이
print("측정값 범위(train):", X_train.agg(lambda c: c.max() - c.min()).round(3).to_dict())

KNN(k=5) 스케일링 전 0.8056 → 후 0.8333
예측이 바뀐 건포도: 16개 / 144
측정값 범위(train): {'MajorAxisLength': 771.662, 'Extent': 0.444}


**결과** · 예측 정답은 16개 · 정확도 0.8056 → 0.8333

- `KNN(k=5) 스케일링 전 0.8056 → 후 0.8333` — 4개 더 맞힘
- `예측이 바뀐 건포도: 16개 / 144` — 판단 기준이 바뀐 건포도는 4개가 아니라 16개 (맞게 바뀐 10 · 틀리게 바뀐 6)
- `측정값 범위(train): {'MajorAxisLength': 771.662, 'Extent': 0.444}` — 범위가 천 배 넘게 다름 → 전에는 길이만 봄

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>정확도만 보기</td><td>올랐다 · 내렸다</td><td>근거로 부족</td></tr>
<tr><td>바뀐 예측 수</td><td>판단 기준이 얼마나 바뀌었나</td><td>서술 2의 근거</td></tr>
<tr><td>점수가 완전히 같으면</td><td>스케일링 안 한 데이터를 넣은 것 (원본 경고)</td><td>변환을 확인</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - scaler를 validation까지 합쳐 fit하면 누수 (파일별 3)

> ✅ **한 줄 정리** · 단위가 크게 다른 두 측정값에서는 스케일링이 KNN의 이웃 관계를 바꾸며, 그 크기는 정확도와 바뀐 예측 수로 함께 말한다.

**딥러닝 코드에서는**

- 입력 정규화 전후로 예측이 얼마나 바뀌는지 보는 것도 좋은 점검

---
## 3. k 고르기 · 최종 1개

> **이 장의 흐름**
>
> - Pipeline + GridSearchCV로 train 내부 CV에서 k 선택 → 고른 k의 validation
> - 그다음 네 후보를 한 표로 모아 규칙(validation 최고 · 동점이면 단순한 쪽)으로 하나

<img src="images/pfm9_k.png" alt="k별 train 내부 CV" width="595">

**파이프라인 위치** · ⑤ 교차검증 · 선택

### 3-1. `A` k를 train 내부 CV로 · `GridSearchCV` · `knn__n_neighbors` <sub>원본 셀 12 ~ 14 (문제 4)</sub>

> 🎯 **k = 5는 그냥 쓴 값인데, train 내부 CV로 고르면 몇이 되는가?**

💡 **쉽게 말하면**

- Pipeline 안에 scaler가 있으면 **fold마다** scaler도 다시 fit.
- 그래서 X_train을 그대로 넣으면 됨.

**실행 흐름**

1. k 후보 정하기
2. GridSearchCV(cv=5)를 train에
3. 고른 k · CV · validation 정확도

**먼저 예측** · 고른 k는 5보다 클까 작을까? — 답은 코드 실행 결과 아래에

In [6]:
# 왜 · K 후보마다 스케일링을 같은 방식으로 묶어 공정하게 비교하기 위해
pipe = Pipeline([("scaler", StandardScaler()), ("knn", KNN())])  # 원본 힌트 2
k_list = [1, 3, 5, 7, 9, 11, 15, 21, 31]                         # 후보 9개 (최소 3개)
# train 안에서만
grid = GridSearchCV(pipe, {"knn__n_neighbors": k_list}, cv=5).fit(X_train, y_train)
best_k = grid.best_params_["knn__n_neighbors"]                   # 고른 k
acc_grid = accuracy_score(y_valid, grid.predict(X_valid))        # 문제 5에서 씀
print(f"고른 k: {best_k} · train CV {grid.best_score_:.4f} · validation {acc_grid:.4f}")  # 문제 4
cv_by_k = pd.Series(grid.cv_results_["mean_test_score"], index=k_list).round(4)  # k별 CV
print("k별 train CV:", cv_by_k.to_dict())                        # 곡선 (3장 그림)

고른 k: 21 · train CV 0.8559 · validation 0.8611
k별 train CV: {1: 0.7847, 3: 0.8282, 5: 0.8177, 7: 0.8403, 9: 0.842, 11: 0.8507, 15: 0.8507, 21: 0.8559, 31: 0.8455}


**결과** · 예측 정답은 큼 — k = 21

- `고른 k: 21 · train CV 0.8559 · validation 0.8611` — train 안에서 고른 k
- `k별 train CV: {1: 0.7847, 3: 0.8282, 5: 0.8177, 7: 0.8403, 9: 0.842, 11: 0.8507, 15: 0.8507, 21: 0.8559, 31: 0.8455}` — k = 1이 가장 낮고 11 ~ 21이 평평하게 높음 · 31에서 다시 내려감

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>k = 5 (그냥 쓴 값)</td><td>근거 없음</td><td>출발점</td></tr>
<tr><td>GridSearchCV로 고른 k</td><td>train 내부 CV 최고</td><td>문제 4의 답</td></tr>
<tr><td>k별 CV 곡선</td><td>동점 · 평평한 구간 확인</td><td>1위를 정답으로 부르지 않기</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - k를 validation으로 고르면 validation이 고르는 데 쓰인 점수가 되어 문제 5의 비교가 흐려짐 → train CV로

> ✅ **한 줄 정리** · k는 scaler를 품은 Pipeline으로 train 내부 CV에서 고르고, 고른 모델의 validation 점수는 후보 비교에만 쓴다.

**딥러닝 코드에서는**

- 하이퍼파라미터는 train(또는 내부 CV)으로, 후보 비교는 val로, 보고는 test로

### 3-2. `A` 네 후보 · 하나 고르기 · `pd.DataFrame` · `선택 규칙` <sub>원본 셀 15 ~ 17 (문제 5)</sub>

> 🎯 **로지스틱 · KNN 스케일링 없음 · KNN 스케일링 · KNN 튜닝 중 무엇을 고르고, 정확도 말고 근거는?**

💡 **쉽게 말하면**

- 표를 보기 **전에** 규칙 선언 · validation 최고, 0.01 안쪽으로 비슷하면 **더 단순한 쪽**.
- 단순함 순서 · 로지스틱 < KNN k=5 < KNN 튜닝.

**실행 흐름**

1. 규칙 선언
2. 네 후보 validation 표
3. 규칙으로 하나 · 근거 2개

**먼저 예측** · 가장 높은 후보와 고르게 될 후보는 같을까? — 답은 코드 실행 결과 아래에

In [7]:
# 왜 · 모든 후보를 같은 validation 점수표에 모아 하나를 고르기 위해
names = ["로지스틱 회귀", "KNN k=5 (스케일링 없음)",  # 원본 셀 16의 네 가지
         "KNN k=5 (스케일링)", f"KNN k={best_k} (튜닝)"]
scores = [acc_lr, acc_knn, acc_knn_s, acc_grid]       # 같은 validation 점수
board = pd.DataFrame({"모델": names, "validation": scores, "단순함": [1, 2, 2, 3]})  # 작을수록 단순
print(board.round(4).to_string(index=False))          # 한 표
top = board["validation"].max()                       # 최고 점수
near = board[board["validation"] >= top - 0.01]       # 최고와 0.01 안쪽
ordered = near.sort_values(["단순함", "validation"], ascending=[True, False])  # 단순한 쪽 먼저
final_name = ordered.iloc[0]["모델"]                  # 규칙대로 하나
print("최종 선택:", final_name)                       # 서술 3

               모델  validation  단순함
          로지스틱 회귀      0.8472    1
KNN k=5 (스케일링 없음)      0.8056    2
   KNN k=5 (스케일링)      0.8333    2
    KNN k=21 (튜닝)      0.8611    3
최종 선택: KNN k=21 (튜닝)


**결과** · 예측 정답은 같음 — KNN k=21 (튜닝)

- `KNN k=21 (튜닝)      0.8611    3` — 최고 · 0.01 안쪽 후보가 이것뿐
- `로지스틱 회귀      0.8472    1` — 가장 단순하지만 최고보다 0.014 낮음 (규칙 밖)
- `최종 선택: KNN k=21 (튜닝)` — 서술 3 · 근거 1 validation 최고 · 근거 2 k를 train CV로 골라 근거가 있음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>근거 1 · validation</td><td>네 후보 중 최고 근처</td><td>정확도</td></tr>
<tr><td>근거 2 · 단순함</td><td>같은 수준이면 더 단순한 쪽</td><td>정확도 외의 근거</td></tr>
<tr><td>다른 근거 예</td><td>바뀐 예측 수 · 기준선 대비 · 설명 가능성</td><td>서술 3에 하나 이상</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 동점이면 성능으로는 못 고름 (4-2) · 규칙은 표를 보기 전에
> - validation 144개에서 0.01 = 약 1.4개 차이

> ✅ **한 줄 정리** · 후보를 한 표로 모은 뒤 미리 정한 규칙(최고와 비슷하면 더 단순한 쪽)으로 하나를 고르고, 정확도 외의 근거를 함께 적는다.

**딥러닝 코드에서는**

- 비슷한 성능이면 더 작은 · 빠른 · 설명 가능한 모델을 고름

---
## 4. 종합 실습 · test 딱 한 번 · 서술 4 ~ 6

> **이 장의 흐름**
>
> - 문제 5에서 고른 모델을 dev(train + validation)로 다시 학습 → test 180개 한 번
> - 그다음 서술 4 ~ 6 · 한계와 다음에 확인할 것은 train CV로만 (모델은 안 바꿈)

<img src="images/pfm9_practice.png" alt="네 후보 validation · 고른 모델의 test 1회" width="595">

**파이프라인 위치** · ⑥ test 1회 → 회고

> 🎯 **test 180개에서 1명 틀리면 정확도가 얼마나 움직이고, validation과의 차이는 얼마인가?**

💡 **쉽게 말하면**

- Pipeline이면 재학습 때 scaler도 dev로 다시 fit.
- 결과를 보고 모델을 바꾸지 않음.

**실행 흐름**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">단계</th><th style="text-align: left;">하는 일</th><th style="text-align: left;">쓰는 문법</th></tr>
</thead>
<tbody>
<tr><td>1</td><td>고른 모델을 Pipeline으로 다시 만들기</td><td>3-2</td></tr>
<tr><td>2</td><td>dev 720으로 fit</td><td>준비</td></tr>
<tr><td>3</td><td>test 180 한 번 · 1명 = 몇 %p</td><td>준비</td></tr>
<tr><td>4</td><td>서술 6 · 7개 전부의 train CV (참고)</td><td>1-2</td></tr>
</tbody></table>

**먼저 예측** · test 점수는 validation보다 높을까 낮을까? — 답은 코드 실행 결과 아래에

In [8]:
# 1 · 2단계: 고른 모델을 Pipeline으로 · dev 720개로 다시 학습
# 왜 · 고른 설정을 바꾸지 않고 train + validation 전체로 다시 학습하기 위해
def scaled_knn(k):                                                    # 스케일러 + KNN
    return Pipeline([("scaler", StandardScaler()), ("knn", KNN(k))])  # fold · 재학습마다 scaler fit


def plain_lr():                                                   # 스케일러 없는 로지스틱
    return Pipeline([("lr", LogisticRegression(max_iter=5000))])  # 원본 설정


# 왜 · 후보마다 새 모델을 만드는 함수를 두어 학습이 서로 섞이지 않게 하기 위해
makers = {"로지스틱 회귀": plain_lr,    # 문제 5 표와 같은 이름
          "KNN k=5 (스케일링 없음)": lambda: Pipeline([("knn", KNN(5))]),
          "KNN k=5 (스케일링)": lambda: scaled_knn(5),
          f"KNN k={best_k} (튜닝)": lambda: scaled_knn(best_k)}
final_model = makers[final_name]()      # 문제 5의 선택 그대로
X_dev, y_dev = dev[PAIR], dev["Class"]  # train + validation
final_model.fit(X_dev, y_dev)           # scaler도 dev로 다시 fit


# 3단계: test 딱 한 번
# 왜 · 봉인해 둔 test를 선택이 끝난 뒤 한 번만 써서 공정한 점수를 얻기 위해
acc_test = accuracy_score(y_test, final_model.predict(test_df[PAIR]))  # 처음이자 마지막
acc_val = board.set_index("모델").loc[final_name, "validation"]        # 고를 때 쓴 점수
diff = acc_test - acc_val                                              # 서술 4의 차이
print(f"최종 모델: {final_name} · validation {acc_val:.4f} · "         # 보고
      f"test {acc_test:.4f} · 차이 {diff:+.4f}")
print(f"test에서 1명 틀릴 때 변화: {1 / len(y_test):.4f}")             # 180개 해상도

최종 모델: KNN k=21 (튜닝) · validation 0.8611 · test 0.8833 · 차이 +0.0222
test에서 1명 틀릴 때 변화: 0.0056


**실행 흐름**

1. 특성 7개 전부와 고른 2개의 train CV 계산
2. 두 점수를 나란히 출력 (모델은 바꾸지 않음)

In [9]:
# 4단계: 서술 6 · 다음에 확인할 것 (train CV만 · 모델은 바꾸지 않음)
# 왜 · test를 본 뒤 모델을 바꾸지 않고, 다음 실험 아이디어만 train CV로 확인하기 위해
all7 = cross_val_score(pair_lr, train_df[FEATURES], y_train, cv=5).mean()  # 7개 전부
two = cross_val_score(pair_lr, X_train, y_train, cv=5).mean()              # 고른 2개
print(f"train CV · 2개 {two:.4f} vs 7개 전부 {all7:.4f}")                  # 정보를 더 쓰면?

train CV · 2개 0.8663 vs 7개 전부 0.8611


**결과** · 예측 정답은 높음 — test 0.8833 (validation 0.8611)

- `최종 모델: KNN k=21 (튜닝) · validation 0.8611 · test 0.8833 · 차이 +0.0222` — 서술 4 · test가 더 높게 나옴
- `test에서 1명 틀릴 때 변화: 0.0056` — 차이 0.0222 = 약 4명 · dev 전체로 다시 학습한 효과도 섞임
- `train CV · 2개 0.8663 vs 7개 전부 0.8611` — 서술 6 · 7개를 다 써도 좋아지지 않음 · 크기 측정값이 겹쳐서

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>서술 4 · validation vs test</td><td>차이와 그 이유 (고를 때 쓴 점수 · 180개 흔들림)</td><td>차이는 보통 작게 흔들림</td></tr>
<tr><td>서술 5 · 한계</td><td>측정값 2개뿐 · 두 품종이 겹치는 구간은 어떤 모델도 못 가름</td><td>데이터의 한계</td></tr>
<tr><td>서술 6 · 다음</td><td>7개 전부 · 다른 조합 · 반복 CV</td><td>train CV로만 확인</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 4단계는 test 뒤의 회고 · 그 결과로 최종 모델을 바꾸지 않음 (바꾸려면 새 test가 필요)
> - 제출 전 확인 · test를 문제 6 전에 열지 않음 · 결과를 보고 바꾸지 않음 · Restart & Run All

> ✅ **한 줄 정리** · 고른 모델은 dev 전체로 다시 학습해 test를 한 번만 평가하고, 이후의 탐색은 train CV로만 해서 선택을 되돌리지 않는다.

**딥러닝 코드에서는**

- 최종 모델은 train + val로 다시 학습 → test 1회 · 이후 실험은 다음 버전으로

---
## 그래서 오늘 무엇을 배웠는가?

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">학습 목표</th><th style="text-align: left;">할 수 있게 된 것</th><th style="text-align: left;">확인 위치</th></tr>
</thead>
<tbody>
<tr><td>1. 측정값 2개</td><td>분리 점수 MajorAxisLength 1위 · Extent 꼴찌인데 조합은 1위(0.8663)</td><td>1-1 · 1-2</td></tr>
<tr><td>2. 모델 비교</td><td>로지스틱 0.8472 · KNN 0.8056 · 기준선 0.5</td><td>2-1</td></tr>
<tr><td>3. 스케일링</td><td>0.8056 → 0.8333 · 바뀐 예측 16개</td><td>2-2</td></tr>
<tr><td>4. k · 선택</td><td>k = 21 (train CV) · 규칙으로 KNN k=21</td><td>3-1 · 3-2</td></tr>
<tr><td>5. test 1회</td><td>test 0.8833 · 1명 = 0.0056 · 7개 전부는 0.8611로 이득 없음</td><td>4</td></tr>
</tbody></table>

### 주제별 한 줄 · 반드시 기억할 것

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">등급</th><th style="text-align: left;">주제</th><th style="text-align: left;">반드시 기억할 것</th><th style="text-align: left;">딥러닝 코드에서</th></tr>
</thead>
<tbody>
<tr><td><code>A</code></td><td>train으로만 특성 선택</td><td>분리 점수 · 조합 CV</td><td>채널 선택도 train에서</td></tr>
<tr><td><code>A</code></td><td>기준선 0.5</td><td>반반 찍기 · 넘어야 할 선</td><td>첫 기준선</td></tr>
<tr><td><code>A</code></td><td>스케일링 · 바뀐 예측 수</td><td>정확도 외의 근거</td><td>정규화 전후 비교</td></tr>
<tr><td><code>A</code></td><td>Pipeline + GridSearchCV</td><td>fold마다 scaler 다시 fit</td><td>하이퍼파라미터 탐색</td></tr>
<tr><td><code>A</code></td><td>선택 규칙</td><td>표를 보기 전에 · 비슷하면 단순한 쪽</td><td>작은 모델 선호</td></tr>
<tr><td><code>A</code></td><td>test 1회</td><td>dev 재학습 → 한 번 · 바꾸지 않음</td><td>test loader 1회</td></tr>
<tr><td><code>B</code></td><td>상관된 측정값</td><td>정보가 겹쳐 함께 골라도 이득이 작음</td><td>중복 채널</td></tr>
<tr><td><code>B</code></td><td>180개 해상도</td><td>1명 = 0.0056</td><td>작은 test 흔들림</td></tr>
</tbody></table>

---
## 기억 확인 문제

위 내용을 **보지 않고** 답한 뒤, ✅ 한 줄 정리와 비교해 ○ (정확) · △ (일부) · × (모름)를 표시. △ · ×인 섹션만 다시 보기.

### 1. 질문에 답하기

*섹션 → 🎯 질문*

- **1-1** · 측정값을 고를 때 무엇을 보는가?
- **1-2** · 2개 조합은 무엇으로 고르는가?
- **2-1** · 두 모델을 무엇과 비교하는가?
- **2-2** · 스케일링 효과의 근거 두 가지는?
- **3-1** · k는 무엇으로 고르는가?
- **3-2** · 최종 1개의 근거는?
- **4** · test 뒤에 7개 전부가 더 좋으면 바꾸나?

#### 내 답 · 이 셀을 더블클릭해서 작성

**1-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**4** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

### 2. 코드로 답하기

아래 빈 셀에 직접 작성 후 실행.

1. 고른 2개 대신 `["Perimeter", "Eccentricity"]`로 스케일러 + 로지스틱의 train 내부 CV를 소수 넷째 자리로 출력
2. 같은 조합으로 train에 fit해 validation 정확도를 소수 넷째 자리로 출력

In [10]:
# 코드 문제 직접 작성

### 3. 변형 문제 · 조건을 바꿔 다시 풀기

1. 1-2에서 로지스틱 대신 스케일러 + KNN(k=5)으로 조합 순위를 매기면 1위가 같을까?
2. 2-2에서 MinMaxScaler를 쓰면 바뀐 예측 수는?
3. 3-1 후보를 [3, 5, 7]로만 주면 고른 k는?

In [11]:
# 변형 문제 직접 작성

### 4. 모범 답안 · 1 ~ 3을 다 푼 뒤에 보기

#### 질문 답

*섹션 → 답*

- **1-1** · train에서 두 품종이 덜 겹치는지 (평균 차이 ÷ 표준편차)
- **1-2** · train 내부 CV · MajorAxisLength + Extent 1위
- **2-1** · 같은 validation · 기준선 0.5
- **2-2** · 정확도 변화 · 예측이 바뀐 개수
- **3-1** · Pipeline + GridSearchCV · train 내부 CV
- **3-2** · validation 최고 근처 · 같은 수준이면 더 단순한 쪽
- **4** · 안 바꿈 · 바꾸려면 새 test

#### 코드 문제 답

```python
alt = ["Perimeter", "Eccentricity"]
print(round(cross_val_score(pair_lr, train_df[alt], y_train, cv=5).mean(), 4))  # 0.8576
pair_lr.fit(train_df[alt], y_train)
print(round(accuracy_score(y_valid, pair_lr.predict(valid_df[alt])), 4))  # 0.8542
```

#### 변형 문제 답

*문제 → 답*

- **1** · 아님 — KNN 기준 1위는 Area + Perimeter(0.8507) · 모델이 바뀌면 좋은 조합도 바뀜
- **2** · 18개 바뀜 · 정확도 0.8194 — Standard(16개 · 0.8333)와 비슷하지만 같지 않음
- **3** · k = 7 — 후보 범위가 좁으면 그 안에서만 고름 (GridSearch는 탐색기)

---
## 오답 노트

틀린 문제는 정답을 베끼지 말고 **원인**을 적음. 다음 복습 때 이 표의 "재발 방지 질문"부터 스스로에게 물어보기.

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">틀린 내용</th><th style="text-align: left;">틀린 이유</th><th style="text-align: left;">올바른 원칙</th><th style="text-align: left;">재발 방지 질문</th></tr>
</thead>
<tbody>
<tr><td>(예) validation에서 k를 고르고 그 점수로 다시 후보를 비교함</td><td>고르는 데 쓴 점수를 비교에도 씀</td><td>k는 train CV · 후보 비교는 validation · 보고는 test</td><td>이 점수를 이미 고르는 데 썼나?</td></tr>
<tr><td></td><td></td><td></td><td></td></tr>
<tr><td></td><td></td><td></td><td></td></tr>
</tbody></table>

---
## 면접 30초 답변

**Q1. 특성을 몇 개만 골라야 할 때 어떻게 고르나요?**
> - train 데이터 안에서만 근거를 만듭니다.
> - 측정값마다 두 클래스의 평균 차이를 표준편차로 나눠 덜 겹치는 후보를 보고, 조합 자체를 train 내부 교차검증으로 비교합니다.
> - 건포도에서는 크기와 모양을 하나씩 묶은 MajorAxisLength와 Extent가 1위였고, 상관이 큰 두 크기 측정값은 함께 골라도 이득이 작았습니다.

**Q2. 후보 모델 여러 개 중 하나는 어떻게 고르나요?**
> - 표를 보기 전에 규칙을 정합니다.
> - validation 최고와 비슷한 수준이면 더 단순한 쪽을 고르고, 정확도 외에 스케일링 전후 바뀐 예측 수나 기준선 대비 같은 근거를 함께 적습니다.
> - 하이퍼파라미터는 train 내부 CV로, 후보 비교는 validation으로, 보고는 test 한 번으로 역할을 나눕니다.

---
## 최종 요약표

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>train_test_split(stratify=)</code></td><td>sklearn.model_selection</td><td>train · validation · test 3분할</td><td>0 <code>train_test_split(dev, test_size=0.2, …)</code></td></tr>
<tr><td><code>groupby().mean()</code> · <code>std()</code></td><td>pandas</td><td>품종별 평균 · 흩어짐</td><td>1-1 <code>train_df.groupby("Class")[FEATURES]</code></td></tr>
<tr><td><code>itertools.combinations</code></td><td>Python itertools</td><td>2개 조합 21개</td><td>1-2 <code>itertools.combinations(FEATURES, 2)</code></td></tr>
<tr><td><code>cross_val_score</code></td><td>sklearn.model_selection</td><td>train 내부 CV</td><td>1-2 <code>cross_val_score(pair_lr, …, cv=5)</code></td></tr>
<tr><td><code>LogisticRegression</code> · <code>KNeighborsClassifier</code></td><td>sklearn</td><td>두 모델</td><td>2-1 <code>LogisticRegression(max_iter=5000)</code></td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>StandardScaler</code> · <code>fit_transform</code> · <code>transform</code></td><td>sklearn.preprocessing</td><td>스케일링 (train에만 fit)</td><td>2-2 <code>sc.fit_transform(X_train)</code></td></tr>
<tr><td><code>GridSearchCV</code> · <code>Pipeline</code></td><td>sklearn</td><td>k 탐색 · fold마다 scaler</td><td>3-1 <code>GridSearchCV(pipe, {…}, cv=5)</code></td></tr>
<tr><td><code>accuracy_score</code></td><td>sklearn.metrics</td><td>정확도</td><td>4 <code>accuracy_score(y_test, …)</code></td></tr>
</tbody></table>

### 할 것 · 하지 말 것

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">✅ 할 것</th><th style="text-align: left;">⚠️ 하지 말 것</th></tr>
</thead>
<tbody>
<tr><td>특성 선택은 train으로만</td><td>validation · test를 보고 특성 고르기</td></tr>
<tr><td>scaler는 train에만 fit</td><td>validation까지 합쳐 fit</td></tr>
<tr><td>k는 train 내부 CV로</td><td>validation으로 k를 고르고 같은 점수로 후보 비교</td></tr>
<tr><td>규칙을 표보다 먼저</td><td>결과를 보고 규칙 정하기</td></tr>
<tr><td>test는 마지막 1회</td><td>test 결과를 보고 모델 바꾸기</td></tr>
</tbody></table>

### 헷갈리는 짝 한눈에 보기

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">짝</th><th style="text-align: left;">차이</th><th style="text-align: left;">결론</th></tr>
</thead>
<tbody>
<tr><td>train CV vs validation</td><td>하이퍼파라미터 · 특성 vs 후보 비교</td><td>역할을 나눔</td></tr>
<tr><td>정확도 vs 바뀐 예측 수</td><td>결과 vs 판단 기준의 변화</td><td>둘 다 근거</td></tr>
<tr><td>dev 재학습 vs train만</td><td>720개 vs 576개</td><td>최종 모델은 dev로</td></tr>
<tr><td>test 뒤 회고 vs 재선택</td><td>train CV로 다음 질문 vs 모델 바꾸기</td><td>회고만</td></tr>
</tbody></table>

---
## 다음 복습일

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">언제</th><th style="text-align: left;">할 일</th><th style="text-align: left;">걸리는 시간</th></tr>
</thead>
<tbody>
<tr><td>내일</td><td>기억 확인 문제 1을 자료 없이 다시 답하고 오답 노트 갱신</td><td>10분</td></tr>
<tr><td>3일 뒤</td><td>변형 문제 3개 · 원본 정답 · 상세 설명편과 내 답 비교</td><td>30분</td></tr>
<tr><td>7일 뒤</td><td>다른 조합(예: 2위 조합)으로 문제 2 ~ 6을 빈 셀에 다시</td><td>30분</td></tr>
<tr><td>14일 뒤</td><td>오답 노트에 남은 문제만 · 보강 3(앙상블 ②) 시작 전 ✅ 줄만 읽기</td><td>10분</td></tr>
</tbody></table>

---
## 부록

### 이 노트북을 만든 방법

- 원본 빈칸 과제(문제 1 ~ 6 · 서술 1 ~ 6)를 직접 풀어 1 ~ 3장 · 6개 섹션 · 4장 종합 실습으로 정리
- 원본의 접힌 힌트(details 태그)는 형식 규칙상 각 섹션의 💡 · 실행 흐름으로 옮김
- 문제 1은 산점도 대신 train 내부 근거(분리 점수 · 조합 21개 CV)로 고름 · 원본 plot_pair 그림은 1장 그림으로
- 정답 · 상세 설명편은 이 풀이 뒤에 비교하도록 보존 (이 노트북은 그 파일을 쓰지 않음)

### 읽는 법

*표시 → 의미*

- **`A` · `B`** · **A** 딥러닝 코드에 바로 나오는 필수 · **B** 알아두면 되는 것
- **🎯** · 이번 섹션에서 답해야 할 질문
- **💡** · 쉬운 설명 · 비유
- **⚠️** · 자주 하는 실수
- **✅** · 기억할 결론 = 🎯 질문의 답

- **섹션 앞부분** · 🎯 질문 → 💡 쉬운 설명 → 실행 흐름 → 먼저 예측 → 코드
- **섹션 뒷부분** · 결과(예측 정답 · 출력 줄) → 차이 · 언제 쓰나 → ⚠️ 주의 → ✅ 한 줄 정리
- **코드 주석** · 코드 위는 이 단계에서 하는 일, 코드 옆은 그 줄이 필요한 이유 (줄마다 · 열 맞춤)

### 원본 → 섹션 대응표

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">원본</th><th style="text-align: left;">내용</th><th style="text-align: left;">이 노트북</th></tr>
</thead>
<tbody>
<tr><td>셀 0</td><td>과제 소개 · 문제 6개 · 지켜야 할 것 · 참고</td><td>0. 원본 과제 · 힌트</td></tr>
<tr><td>셀 1 · 2</td><td>준비 (분할 · plot_pair)</td><td>0. 준비 코드 · 1장 그림</td></tr>
<tr><td>셀 3 ~ 5</td><td>문제 1 · 힌트 · 서술 1</td><td>1-1 · 1-2</td></tr>
<tr><td>셀 6 ~ 8</td><td>문제 2 · 힌트</td><td>2-1</td></tr>
<tr><td>셀 9 ~ 11</td><td>문제 3 · 힌트 2개 · 서술 2</td><td>2-2 · 2장 그림</td></tr>
<tr><td>셀 12 ~ 14</td><td>문제 4 · 힌트 2개</td><td>3-1 · 3장 그림</td></tr>
<tr><td>셀 15 ~ 17</td><td>문제 5 · 힌트 · 서술 3</td><td>3-2</td></tr>
<tr><td>셀 18 ~ 20</td><td>문제 6 · 힌트</td><td>4. 종합 실습 1 ~ 3단계</td></tr>
<tr><td>셀 21</td><td>서술 4 ~ 6 · 제출 전 확인</td><td>4. 종합 실습 4단계 · 주의</td></tr>
</tbody></table>

### 전체 코드

1 ~ 4장 코드를 장별 셀로 모음. **위에서부터 이 셀들만 실행해도 전체가 동작**하며, 각 블록 머리 주석의 번호가 섹션.

**실행 흐름**

1. 1-1 품종별 특성 차이
2. 1-2 특성 두 개 조합 순위

In [12]:
# ---------- 1-1 ----------
# 왜 · 두 품종을 가장 잘 가르는 특성을 평균 차이 · 퍼짐으로 미리 살펴보기 위해
g = train_df.groupby("Class")[FEATURES]                        # train만 · 품종별
gap = (g.mean().loc["Besni"] - g.mean().loc["Kecimen"]).abs()  # 평균 차이
spread = g.std().mean()                                        # 두 품종 표준편차의 평균
sep = (gap / spread).sort_values(ascending=False)              # 클수록 덜 겹침
print(sep.round(3).to_string())                                # 분리 점수
# 거의 같은 정보
print("크기 측정값끼리 상관:", round(train_df[["Area", "ConvexArea"]].corr().iloc[0, 1], 3))


# ---------- 1-2 ----------
# 왜 · 특성 두 개 조합마다 같은 모델로 점수를 비교해 힌트의 근거를 찾기 위해
lr5000 = LogisticRegression(max_iter=5000)                                # 원본 설정
pair_lr = Pipeline([("scaler", StandardScaler()), ("lr", lr5000)])        # 조합 비교용
rows = []                                                                 # 조합마다
for a, b in itertools.combinations(FEATURES, 2):                          # 7개 중 2개 = 21개
    s = cross_val_score(pair_lr, train_df[[a, b]], y_train, cv=5).mean()  # train 안에서만
    rows.append([f"{a} + {b}", s])                                        # 한 줄
pairs = pd.DataFrame(rows, columns=["조합", "train CV"])                  # 21행
pairs = pairs.sort_values("train CV", ascending=False)                    # 1위부터
print(pairs.head(5).round(4).to_string(index=False))                      # 위 5개
# 힌트 조합 순위
hint_rank = int((pairs["조합"] == "Eccentricity + Perimeter").to_numpy().argmax()) + 1
print("힌트 조합(Perimeter + Eccentricity) 순위:", hint_rank, "/ 21")     # 참고
PAIR = pairs.iloc[0]["조합"].split(" + ")                                 # 1위 조합 고정
X_train, X_valid = train_df[PAIR], valid_df[PAIR]                         # 뒤 문제에서 계속 씀
print("고른 측정값:", PAIR)                                               # 문제 1 답

MajorAxisLength    1.916
Perimeter          1.847
Area               1.753
ConvexArea         1.745
MinorAxisLength    1.201
Eccentricity       1.009
Extent             0.264
크기 측정값끼리 상관: 0.995


                          조합  train CV
    MajorAxisLength + Extent    0.8663
   Eccentricity + ConvexArea    0.8611
          Extent + Perimeter    0.8593
MajorAxisLength + ConvexArea    0.8576
 MajorAxisLength + Perimeter    0.8576
힌트 조합(Perimeter + Eccentricity) 순위: 6 / 21
고른 측정값: ['MajorAxisLength', 'Extent']


**실행 흐름**

1. 2-1 로지스틱 기준선
2. 2-2 train에만 fit한 스케일링

In [13]:
# ---------- 2-1 ----------
# 왜 · 스케일링 없이 기준 모델의 validation 점수를 먼저 남기기 위해
lr = LogisticRegression(max_iter=5000).fit(X_train, y_train)                    # 원본 힌트 그대로
lr_pred = lr.predict(X_valid)                                                   # validation 예측
knn = KNN(n_neighbors=5).fit(X_train, y_train)                                  # 스케일링 안 함
knn_pred = knn.predict(X_valid)                                                 # 문제 3에서 비교
acc_lr, acc_knn = accuracy_score(y_valid, lr_pred), accuracy_score(y_valid, knn_pred)  # 두 점수
print(f"로지스틱 validation: {acc_lr:.4f} · 기준선보다 +{acc_lr - 0.5:.4f}")    # 문제 2
print(f"KNN(k=5) validation: {acc_knn:.4f} · 기준선보다 +{acc_knn - 0.5:.4f}")  # 문제 2


# ---------- 2-2 ----------
# 왜 · train에만 fit해 validation · test 정보가 새지 않게 스케일링하기 위해
sc = StandardScaler()                                              # 평균 0 · 표준편차 1
X_train_s = sc.fit_transform(X_train)                              # train에만 fit
X_valid_s = sc.transform(X_valid)                                  # validation은 transform만
knn_s = KNN(n_neighbors=5).fit(X_train_s, y_train)                 # 넣는 데이터만 바뀜
pred_s = knn_s.predict(X_valid_s)                                  # 스케일링 후 예측
acc_knn_s = accuracy_score(y_valid, pred_s)                        # 정확도
print(f"KNN(k=5) 스케일링 전 {acc_knn:.4f} → 후 {acc_knn_s:.4f}")  # 서술 2
print(f"예측이 바뀐 건포도: {(knn_pred != pred_s).sum()}개 / {len(pred_s)}")  # 정확도 말고 근거
# 단위 차이
print("측정값 범위(train):", X_train.agg(lambda c: c.max() - c.min()).round(3).to_dict())

로지스틱 validation: 0.8472 · 기준선보다 +0.3472
KNN(k=5) validation: 0.8056 · 기준선보다 +0.3056
KNN(k=5) 스케일링 전 0.8056 → 후 0.8333
예측이 바뀐 건포도: 16개 / 144
측정값 범위(train): {'MajorAxisLength': 771.662, 'Extent': 0.444}


**실행 흐름**

1. 3-1 K 후보 비교 (train 안에서만)
2. 3-2 후보 점수표

In [14]:
# ---------- 3-1 ----------
# 왜 · K 후보마다 스케일링을 같은 방식으로 묶어 공정하게 비교하기 위해
pipe = Pipeline([("scaler", StandardScaler()), ("knn", KNN())])  # 원본 힌트 2
k_list = [1, 3, 5, 7, 9, 11, 15, 21, 31]                         # 후보 9개 (최소 3개)
# train 안에서만
grid = GridSearchCV(pipe, {"knn__n_neighbors": k_list}, cv=5).fit(X_train, y_train)
best_k = grid.best_params_["knn__n_neighbors"]                   # 고른 k
acc_grid = accuracy_score(y_valid, grid.predict(X_valid))        # 문제 5에서 씀
print(f"고른 k: {best_k} · train CV {grid.best_score_:.4f} · validation {acc_grid:.4f}")  # 문제 4
cv_by_k = pd.Series(grid.cv_results_["mean_test_score"], index=k_list).round(4)  # k별 CV
print("k별 train CV:", cv_by_k.to_dict())                        # 곡선 (3장 그림)


# ---------- 3-2 ----------
# 왜 · 모든 후보를 같은 validation 점수표에 모아 하나를 고르기 위해
names = ["로지스틱 회귀", "KNN k=5 (스케일링 없음)",  # 원본 셀 16의 네 가지
         "KNN k=5 (스케일링)", f"KNN k={best_k} (튜닝)"]
scores = [acc_lr, acc_knn, acc_knn_s, acc_grid]       # 같은 validation 점수
board = pd.DataFrame({"모델": names, "validation": scores, "단순함": [1, 2, 2, 3]})  # 작을수록 단순
print(board.round(4).to_string(index=False))          # 한 표
top = board["validation"].max()                       # 최고 점수
near = board[board["validation"] >= top - 0.01]       # 최고와 0.01 안쪽
ordered = near.sort_values(["단순함", "validation"], ascending=[True, False])  # 단순한 쪽 먼저
final_name = ordered.iloc[0]["모델"]                  # 규칙대로 하나
print("최종 선택:", final_name)                       # 서술 3

고른 k: 21 · train CV 0.8559 · validation 0.8611
k별 train CV: {1: 0.7847, 3: 0.8282, 5: 0.8177, 7: 0.8403, 9: 0.842, 11: 0.8507, 15: 0.8507, 21: 0.8559, 31: 0.8455}
               모델  validation  단순함
          로지스틱 회귀      0.8472    1
KNN k=5 (스케일링 없음)      0.8056    2
   KNN k=5 (스케일링)      0.8333    2
    KNN k=21 (튜닝)      0.8611    3
최종 선택: KNN k=21 (튜닝)


**실행 흐름**

1. 고른 모델을 dev 720개로 다시 학습
2. test 딱 한 번
3. 다음 실험 아이디어 확인

In [15]:
# ---------- 4. 종합 실습 ----------
# 1 · 2단계: 고른 모델을 Pipeline으로 · dev 720개로 다시 학습
# 왜 · 고른 설정을 바꾸지 않고 train + validation 전체로 다시 학습하기 위해
def scaled_knn(k):                                                    # 스케일러 + KNN
    return Pipeline([("scaler", StandardScaler()), ("knn", KNN(k))])  # fold · 재학습마다 scaler fit


def plain_lr():                                                   # 스케일러 없는 로지스틱
    return Pipeline([("lr", LogisticRegression(max_iter=5000))])  # 원본 설정


# 왜 · 후보마다 새 모델을 만드는 함수를 두어 학습이 서로 섞이지 않게 하기 위해
makers = {"로지스틱 회귀": plain_lr,    # 문제 5 표와 같은 이름
          "KNN k=5 (스케일링 없음)": lambda: Pipeline([("knn", KNN(5))]),
          "KNN k=5 (스케일링)": lambda: scaled_knn(5),
          f"KNN k={best_k} (튜닝)": lambda: scaled_knn(best_k)}
final_model = makers[final_name]()      # 문제 5의 선택 그대로
X_dev, y_dev = dev[PAIR], dev["Class"]  # train + validation
final_model.fit(X_dev, y_dev)           # scaler도 dev로 다시 fit


# 3단계: test 딱 한 번
# 왜 · 봉인해 둔 test를 선택이 끝난 뒤 한 번만 써서 공정한 점수를 얻기 위해
acc_test = accuracy_score(y_test, final_model.predict(test_df[PAIR]))  # 처음이자 마지막
acc_val = board.set_index("모델").loc[final_name, "validation"]        # 고를 때 쓴 점수
diff = acc_test - acc_val                                              # 서술 4의 차이
print(f"최종 모델: {final_name} · validation {acc_val:.4f} · "         # 보고
      f"test {acc_test:.4f} · 차이 {diff:+.4f}")
print(f"test에서 1명 틀릴 때 변화: {1 / len(y_test):.4f}")             # 180개 해상도


# 4단계: 서술 6 · 다음에 확인할 것 (train CV만 · 모델은 바꾸지 않음)
# 왜 · test를 본 뒤 모델을 바꾸지 않고, 다음 실험 아이디어만 train CV로 확인하기 위해
all7 = cross_val_score(pair_lr, train_df[FEATURES], y_train, cv=5).mean()  # 7개 전부
two = cross_val_score(pair_lr, X_train, y_train, cv=5).mean()              # 고른 2개
print(f"train CV · 2개 {two:.4f} vs 7개 전부 {all7:.4f}")                  # 정보를 더 쓰면?

최종 모델: KNN k=21 (튜닝) · validation 0.8611 · test 0.8833 · 차이 +0.0222
test에서 1명 틀릴 때 변화: 0.0056
train CV · 2개 0.8663 vs 7개 전부 0.8611
